# Export Data Review

Use this notebook to inspect one export or your full Mountain Project export catalog. By default it loads every export under `data/exports` and summarizes how much area, route, comment, photo, and route-stats data has been pulled across the selected outputs.

If you want to focus on a single export again, change `SELECT_ALL_OUTPUTS` in Cell 2 to `False`.

In [134]:
from pathlib import Path
import pandas as pd

from mountainproject_scraper.loaders import (
    discover_export_dirs,
    load_exports,
    resolve_data_root,
    resolve_export_root,
    resolve_project_root,
 )
from mountainproject_scraper.queries import (
    comment_audit,
    dataset_counts,
    manifest_summary,
    route_comment_counts as build_route_comment_counts,
    top_routes_by_page_views,
    top_routes_by_ticks,
 )

CWD = Path.cwd().resolve()
PROJECT_ROOT = resolve_project_root(CWD)
DATA_ROOT = resolve_data_root(PROJECT_ROOT)
EXPORT_ROOT = resolve_export_root(PROJECT_ROOT)
ROOT = EXPORT_ROOT

available_outputs = discover_export_dirs(EXPORT_ROOT, project_root=PROJECT_ROOT)
OUTPUT_CANDIDATES = sorted(available_outputs)
PREFERRED_OUTPUTS = [
    'holcomb_valley_pinnacles',
    'holcomb_valley_pinnacles_auth',
    'central_pinnacles',
    'pinnacles_east',
]
EXCLUDED_OUTPUTS = []

SELECT_ALL_OUTPUTS = True
if SELECT_ALL_OUTPUTS:
    TARGET_NAMES = [name for name in OUTPUT_CANDIDATES if name not in EXCLUDED_OUTPUTS]
else:
    preferred_name = next(
        (name for name in PREFERRED_OUTPUTS if name in available_outputs and name not in EXCLUDED_OUTPUTS),
        None,
    )
    fallback_name = next((name for name in OUTPUT_CANDIDATES if name not in EXCLUDED_OUTPUTS), None)
    TARGET_NAMES = [preferred_name or fallback_name] if (preferred_name or fallback_name) else []

TARGET_NAMES = [name for name in TARGET_NAMES if name]
TARGET_NAME = TARGET_NAMES[0] if TARGET_NAMES else None
COMMENT_AUDIT_OUTPUT_NAME = next(
    (name for name in ['holcomb_valley_pinnacles', 'holcomb_valley_pinnacles_auth'] if name in TARGET_NAMES),
    TARGET_NAME,
)

OUTPUT_DIRS = {name: available_outputs[name] for name in TARGET_NAMES}
OUTPUT_DIR = next(iter(OUTPUT_DIRS.values())) if len(OUTPUT_DIRS) == 1 else None
CACHE_DIR = OUTPUT_DIR / '.cache' / 'http' if OUTPUT_DIR else None
STRUCTURED_DIR = OUTPUT_DIR / 'structured' if OUTPUT_DIR else None
DUCKDB_PATH = STRUCTURED_DIR / 'mountainproject.duckdb' if STRUCTURED_DIR else None

{
    'project_root': str(PROJECT_ROOT),
    'data_root': str(DATA_ROOT),
    'export_root': str(EXPORT_ROOT),
    'export_count_found': len(OUTPUT_CANDIDATES),
    'export_count_selected': len(TARGET_NAMES),
    'excluded_outputs': EXCLUDED_OUTPUTS,
    'selected_outputs': TARGET_NAMES,
    'single_output_mode': len(OUTPUT_DIRS) == 1,
    'comment_audit_output': COMMENT_AUDIT_OUTPUT_NAME,
    'duckdb_path': str(DUCKDB_PATH) if DUCKDB_PATH and DUCKDB_PATH.exists() else None,
}

{'project_root': '/Users/peterwilliams/projects/mountainproject',
 'data_root': '/Users/peterwilliams/projects/mountainproject/data',
 'export_root': '/Users/peterwilliams/projects/mountainproject/data/exports',
 'export_count_found': 18,
 'export_count_selected': 18,
 'excluded_outputs': [],
 'selected_outputs': ['arizona',
  'california',
  'central_pinnacles_cache_test_login',
  'eastern_sierra',
  'high_desert',
  'high_sierra',
  'holcomb_valley_pinnacles',
  'joshua_tree_national_park',
  'lake_tahoe',
  'los_angeles_basin',
  'nevada',
  'northeast_california',
  'northwest_california',
  'ohio',
  'sequoia_kings_canyon_np',
  'utah',
  'west_virginia',
  'yosemite_national_park'],
 'single_output_mode': False,
 'comment_audit_output': 'holcomb_valley_pinnacles',
 'duckdb_path': None}

In [130]:
if 'loaded' not in globals():
    print('Run Cell 4 first to load the JSONL tables.')
else:
    audit_output_name = COMMENT_AUDIT_OUTPUT_NAME
    print('Auditing comments for output:', audit_output_name)

    audit = comment_audit(loaded, output_name=audit_output_name)
    truncated_routes = audit.truncated_routes
    truncated_areas = audit.truncated_areas

    print('Truncated route threads:', len(truncated_routes))
    print('Truncated area threads:', len(truncated_areas))
    print('Known missing route comments:', audit.missing_route_comments)
    print('Known missing area comments:', audit.missing_area_comments)

    display(truncated_routes.head(25))
    display(truncated_areas.head(25))

Auditing comments for output: holcomb_valley_pinnacles
Truncated route threads: 0
Truncated area threads: 0
Known missing route comments: 0
Known missing area comments: 0


,route_id,name,comment_count,remaining_comment_count


,area_id,name,comment_count,remaining_comment_count


In [135]:
if 'loaded' in globals() and loaded is not None:
    loaded.close()

loaded = load_exports(
    output_names=TARGET_NAMES or None,
    prefer_names=PREFERRED_OUTPUTS,
    select_all=SELECT_ALL_OUTPUTS,
    export_root=EXPORT_ROOT,
    project_root=PROJECT_ROOT,
    open_duckdb=True,
 )

OUTPUT_DIRS = loaded.output_dirs
TARGET_NAMES = loaded.selected_outputs
TARGET_NAME = TARGET_NAMES[0] if TARGET_NAMES else None
COMMENT_AUDIT_OUTPUT_NAME = next(
    (name for name in ['holcomb_valley_pinnacles', 'holcomb_valley_pinnacles_auth'] if name in TARGET_NAMES),
    TARGET_NAME,
)
OUTPUT_DIR = loaded.output_dir
CACHE_DIR = OUTPUT_DIR / '.cache' / 'http' if OUTPUT_DIR else None
STRUCTURED_DIR = OUTPUT_DIR / 'structured' if OUTPUT_DIR else None
DUCKDB_PATH = (
    loaded.duckdb_path
    if loaded.duckdb_path is not None
    else (STRUCTURED_DIR / 'mountainproject.duckdb' if STRUCTURED_DIR else None)
 )

manifests = loaded.manifests
manifest = manifests[TARGET_NAME] if TARGET_NAME and len(TARGET_NAMES) == 1 else {}
areas = loaded.areas
routes = loaded.routes
comments = loaded.comments
photos = loaded.photos
route_stats_summary = loaded.route_stats_summary
route_stars = loaded.route_stars
route_suggested_ratings = loaded.route_suggested_ratings
route_todos = loaded.route_todos
route_ticks = loaded.route_ticks
duckdb_connection = loaded.duckdb_connection

cache_file_count = sum(
    len(list((output_dir / '.cache' / 'http').glob('*.json')))
    for output_dir in OUTPUT_DIRS.values()
    if (output_dir / '.cache' / 'http').exists()
 )
count_summary = dataset_counts(loaded)

display(count_summary)

{
    'selected_outputs': TARGET_NAMES,
    'comment_audit_output': COMMENT_AUDIT_OUTPUT_NAME,
    'cache_files': cache_file_count,
    'duckdb_available': duckdb_connection is not None,
}

,dataset,rows_loaded,unique_records
0,areas,29610,21808
1,routes,123551,93099
2,comments,241565,183716
3,photos,291692,219391
4,route_stats_summary,123554,93102
5,route_stars,2055002,1534971
6,route_suggested_ratings,744516,568193
7,route_todos,6511011,4717502
8,route_ticks,5944739,4397471


{'selected_outputs': ['arizona',
  'california',
  'central_pinnacles_cache_test_login',
  'eastern_sierra',
  'high_desert',
  'high_sierra',
  'holcomb_valley_pinnacles',
  'joshua_tree_national_park',
  'lake_tahoe',
  'los_angeles_basin',
  'nevada',
  'northeast_california',
  'northwest_california',
  'ohio',
  'sequoia_kings_canyon_np',
  'utah',
  'west_virginia',
  'yosemite_national_park'],
 'comment_audit_output': 'holcomb_valley_pinnacles',
 'cache_files': 0,
 'duckdb_available': False}

In [136]:
summary = manifest_summary(loaded) if 'loaded' in globals() else pd.DataFrame()

def grouped_unique_count(frame: pd.DataFrame, column: str) -> pd.Series:
    if frame.empty or column not in frame.columns or 'output_name' not in frame.columns:
        return pd.Series(dtype='int64')
    return frame.groupby('output_name')[column].nunique()

def grouped_row_count(frame: pd.DataFrame) -> pd.Series:
    if frame.empty or 'output_name' not in frame.columns:
        return pd.Series(dtype='int64')
    return frame.groupby('output_name').size()

if summary.empty:
    coverage = pd.DataFrame()
    overview = pd.DataFrame()
    summary
else:
    coverage = summary[[
        'output_name',
        'start_url',
        'finished_at',
        'auth_mode',
        'full_depth',
        'http_cache_mode',
        'reuse_existing_data',
        'route_workers',
        'route_stats_workers',
    ]].copy()
    coverage['finished_at'] = pd.to_datetime(coverage['finished_at'], errors='coerce', utc=True)

    coverage = coverage.merge(
        grouped_unique_count(areas, 'area_id').rename('areas_loaded'),
        on='output_name',
        how='left',
    )
    coverage = coverage.merge(
        grouped_unique_count(routes, 'route_id').rename('routes_loaded'),
        on='output_name',
        how='left',
    )
    coverage = coverage.merge(
        grouped_row_count(comments).rename('comments_loaded'),
        on='output_name',
        how='left',
    )
    coverage = coverage.merge(
        grouped_row_count(photos).rename('photos_loaded'),
        on='output_name',
        how='left',
    )
    coverage = coverage.merge(
        grouped_unique_count(route_stats_summary, 'route_id').rename('route_stats_routes_loaded'),
        on='output_name',
        how='left',
    )
    coverage = coverage.merge(
        grouped_row_count(route_ticks).rename('route_ticks_loaded'),
        on='output_name',
        how='left',
    )

    numeric_columns = [
        'areas_loaded',
        'routes_loaded',
        'comments_loaded',
        'photos_loaded',
        'route_stats_routes_loaded',
        'route_ticks_loaded',
    ]
    for column in numeric_columns:
        coverage[column] = pd.to_numeric(coverage[column], errors='coerce').fillna(0).astype('int64')

    coverage = coverage.sort_values(['finished_at', 'output_name'], ascending=[False, True]).reset_index(drop=True)

    overview = pd.DataFrame(
        [
            {'metric': 'exports_selected', 'value': len(TARGET_NAMES)},
            {'metric': 'unique_areas_across_selected_exports', 'value': int(areas['area_id'].nunique()) if not areas.empty and 'area_id' in areas.columns else 0},
            {'metric': 'unique_routes_across_selected_exports', 'value': int(routes['route_id'].nunique()) if not routes.empty and 'route_id' in routes.columns else 0},
            {'metric': 'comment_rows_across_selected_exports', 'value': int(len(comments)) if not comments.empty else 0},
            {'metric': 'photo_rows_across_selected_exports', 'value': int(len(photos)) if not photos.empty else 0},
            {'metric': 'unique_route_stats_routes_across_selected_exports', 'value': int(route_stats_summary['route_id'].nunique()) if not route_stats_summary.empty and 'route_id' in route_stats_summary.columns else 0},
            {'metric': 'route_tick_rows_across_selected_exports', 'value': int(len(route_ticks)) if not route_ticks.empty else 0},
            {'metric': 'latest_pull_finished_at', 'value': coverage['finished_at'].max().isoformat() if coverage['finished_at'].notna().any() else None},
        ]
    )

    display(overview)
    display(
        coverage[[
            'output_name',
            'finished_at',
            'auth_mode',
            'full_depth',
            'areas_loaded',
            'routes_loaded',
            'comments_loaded',
            'photos_loaded',
            'route_stats_routes_loaded',
            'route_ticks_loaded',
        ]]
    )

    coverage

,metric,value
0,exports_selected,18
1,unique_areas_across_selected_exports,21808
2,unique_routes_across_selected_exports,93099
3,comment_rows_across_selected_exports,241565
4,photo_rows_across_selected_exports,291692
5,unique_route_stats_routes_across_selected_exports,93102
6,route_tick_rows_across_selected_exports,5944739
7,latest_pull_finished_at,2026-06-26T18:39:20.021162+00:00


,output_name,finished_at,auth_mode,full_depth,areas_loaded,routes_loaded,comments_loaded,photos_loaded,route_stats_routes_loaded,route_ticks_loaded
0,arizona,2026-06-26 18:39:20.021162+00:00,login,True,3083,15087,30215,33788,15087,392064
1,utah,2026-06-26 08:46:05.363274+00:00,login,True,4040,20066,48520,52186,20066,1196396
2,ohio,2026-06-25 18:50:06.505540+00:00,login,True,458,2716,1687,4765,2716,45485
3,nevada,2026-06-25 17:36:12.527882+00:00,login,True,1745,7330,18779,18386,7330,739146
4,california,2026-06-25 06:50:05.928742+00:00,login,True,12471,47803,83985,109659,47803,1978331
5,northeast_california,2026-06-24 16:01:55.431498+00:00,login,True,442,1389,937,2072,1389,7498
6,northwest_california,2026-06-24 15:34:12.978745+00:00,login,True,241,1116,1425,2149,1116,12327
7,high_desert,2026-06-24 15:10:31.569039+00:00,login,True,790,2155,2100,4601,2155,70715
8,high_sierra,2026-06-24 04:34:14.392475+00:00,login,True,250,652,1978,4435,652,18358
9,lake_tahoe,2026-06-23 22:18:16.223170+00:00,login,True,1273,4515,8819,8689,4515,224906


In [127]:
if duckdb_connection is None:
    pd.DataFrame(
        {
            'selected_outputs': [', '.join(TARGET_NAMES)],
            'single_output_mode': [len(OUTPUT_DIRS) == 1],
            'duckdb_available': [False],
            'note': ['DuckDB preview is only available when one output is selected and that output has a structured DuckDB database. Set SELECT_ALL_OUTPUTS = False in Cell 2 to narrow to one export.'],
        }
    )
else:
    duckdb_connection.sql("SHOW TABLES").df()

In [124]:
if 'loaded' not in globals():
    pd.DataFrame()
else:
    top_routes_by_ticks(
        loaded,
        output_name=TARGET_NAME if len(TARGET_NAMES) == 1 else None,
        limit=25,
    )

In [125]:
if 'loaded' not in globals():
    pd.DataFrame()
else:
    top_routes_by_page_views(
        loaded,
        output_name=TARGET_NAME if len(TARGET_NAMES) == 1 else None,
        limit=25,
    )

In [126]:
route_comment_table = (
    build_route_comment_counts(
        loaded,
        output_name=TARGET_NAME if len(TARGET_NAMES) == 1 else None,
    )
    if 'loaded' in globals()
    else pd.DataFrame()
 )

total_comments_loaded = int(len(comments)) if not comments.empty else 0
route_comments_loaded = int(route_comment_table['comment_rows'].sum()) if not route_comment_table.empty else 0
area_comments_loaded = int(total_comments_loaded - route_comments_loaded)
routes_with_comments = int((route_comment_table['comment_rows'] > 0).sum()) if not route_comment_table.empty else 0
routes_with_count_mismatch = int((route_comment_table['comment_delta'] != 0).sum()) if not route_comment_table.empty else 0

print(f"Routes loaded: {len(route_comment_table):,}")
print(f"Total comments loaded: {total_comments_loaded:,}")
print(f"Route comments loaded: {route_comments_loaded:,}")
print(f"Area comments loaded: {area_comments_loaded:,}")
print(f"Routes with at least one comment: {routes_with_comments:,}")
print(f"Routes where comment_count != joined comment rows: {routes_with_count_mismatch:,}")

route_comment_table

Routes loaded: 637
Total comments loaded: 1,026
Route comments loaded: 942
Area comments loaded: 84
Routes with at least one comment: 274
Routes where comment_count != joined comment rows: 0


,output_name,route_id,name,yds_grade,route_type_raw,page_views_total,comment_count,comments_truncated,comment_rows,comment_delta
0,holcomb_valley_pinnacles,105985052,Psychedelic Sluice,5.6 YDS,"Trad, Alpine",10046,16,False,16,0
1,holcomb_valley_pinnacles,105825537,Coyotes in the Henhouse,5.10d YDS,"Sport, Alpine",11960,15,False,15,0
2,holcomb_valley_pinnacles,105805463,Bye Crackie,5.7 YDS,"Sport, Alpine",17464,14,False,14,0
3,holcomb_valley_pinnacles,105852331,The Incinerator,5.12a YDS,"Sport, Alpine",15384,14,False,14,0
4,holcomb_valley_pinnacles,105805473,Coyotes at Sunset,5.8 YDS,"Sport, Alpine",15458,13,False,13,0
...,...,...,...,...,...,...,...,...,...,...
632,holcomb_valley_pinnacles,203171968,Standing Ovation,V1 YDS,"Boulder, Alpine",16,0,False,0,0
633,holcomb_valley_pinnacles,203175974,Nucleus Stand-Start,V3 YDS,"Boulder, Alpine",14,0,False,0,0
634,holcomb_valley_pinnacles,203171974,Search for the One,V3 YDS,"Boulder, Alpine",13,0,False,0,0
635,holcomb_valley_pinnacles,203171943,Right Mirror Mantle,V1 YDS,"Boulder, Alpine",11,0,False,0,0


In [111]:
comments[['parent_type', 'parent_id', 'author_name', 'posted_at', 'beta_count', 'body']].head(20) if not comments.empty else comments

,parent_type,parent_id,author_name,posted_at,beta_count,body
0,area,105805238,Chris Owen,"Sep 4, 2006",0,One of my favorite areas locally in SoCal. I h...
1,area,105805238,Adam Stackhouse,"Apr 28, 2007",1,"With a high concentration of well protected, m..."
2,area,105805238,Jordan Ramey,"Aug 6, 2007",0,Van Dusen Canyon Approach speedometer marks: A...
3,area,105805238,SteezeMeggie,"Apr 13, 2010",0,I hiked from Van Deusen Canyon in Big Bear Cit...
4,area,105805238,Jack Howard,"Mar 27, 2011",0,one of the best sport areas in socal. a must f...
5,area,105805238,RAZORsharp,"Jun 27, 2011",0,The North Parking lot is riddled with camp fir...
6,area,105805238,Chris Owen,"Jul 4, 2011",0,I'll second that. 4x4ers and climbers alike. A...
7,area,105805238,Chris Owen,"Sep 1, 2011",0,Derrick you can get pretty darn close if you h...
8,area,105805238,Joseph Stover,"Jul 9, 2012",0,"Not sure how the 4x4 road is, but it looked po..."
9,area,105805238,Paul Doherty,"Nov 25, 2012",0,Here is a map I made for the area - should be ...


In [70]:
photos[['parent_type', 'parent_id', 'photo_id', 'title', 'thumbnail_url', 'image_url', 'local_path']].head(20) if not photos.empty else photos

,parent_type,parent_id,photo_id,title,thumbnail_url,image_url,local_path
0,area,105931166,106478251,An overview map with some climbs to get your be…,https://mountainproject.com/assets/photos/clim...,NaN,NaN
1,area,105931166,106018573,Baby rattler just off the trail near the Claim…,https://mountainproject.com/assets/photos/clim...,NaN,NaN
2,area,105931166,106837355,Overview for some of the Central Pinnacles crag…,https://mountainproject.com/assets/photos/clim...,NaN,NaN
3,area,105931166,106472452,"Old marker for the nearby Mammoth Mine, Holcomb…",https://mountainproject.com/assets/photos/clim...,NaN,NaN
4,area,105931166,107985888,"Central Pinnacles from the west, Holcomb Valley…",https://mountainproject.com/assets/photos/clim...,NaN,NaN
5,area,105931166,107987061,"Obi Midway up Hidden Gold (5.7), Holcomb Valley…",https://mountainproject.com/assets/photos/clim...,NaN,NaN
6,area,105931166,106037111,"Nearing the top of Powder Keg (5.10a), Holcomb…",https://mountainproject.com/assets/photos/clim...,NaN,NaN
7,area,105931166,119325471,Watch out for rattlers. Encountered this guy ne…,https://mountainproject.com/assets/photos/clim...,NaN,NaN
8,area,105931166,114565088,Central Pinnacles from the northern parking are…,https://mountainproject.com/assets/photos/clim...,NaN,NaN
9,area,105931166,107321079,"Relics from the past, Holcomb Valley Pinnacles",https://mountainproject.com/assets/photos/clim...,NaN,NaN


In [71]:
if not routes.empty:
    routes.assign(has_coords=routes['latitude'].notna() & routes['longitude'].notna()).groupby('has_coords').size()
else:
    routes

In [72]:
display(routes.head(100) if not routes.empty else routes)
display(areas.head(100) if not areas.empty else areas)

,output_name,route_id,url,name,breadcrumbs,breadcrumb_urls,grade_raw,yds_grade,route_type_raw,type_tags,...,description,protection,location,details_raw,photos,comments,comment_count,comments_truncated,remaining_comment_count,photo_count
0,central_pinnacles,105878952,https://www.mountainproject.com/route/10587895...,Gold Standard,"[All Locations, California, San Bernardino Mou...","[https://www.mountainproject.com/route-guide, ...",5.6 YDS 4c French 14 Ewbanks V UIAA 12 ZA S 4b...,5.6 YDS,"Sport, Alpine","[Sport, Alpine]",...,Incut plates and edges past two bolts lead to ...,"7 bolts, chain anchors\n(all bolts 1/2"" SS)\nB...",Located on the far right side of the Gold Wall...,"{'Type': 'Sport, Alpine, 80 ft (24 m)', 'GPS':...","[{'photo_id': '106816848', 'parent_type': 'rou...","[{'comment_id': '105900371', 'parent_type': 'r...",11,True,8.0,12
1,central_pinnacles,105805463,https://www.mountainproject.com/route/10580546...,Bye Crackie,"[All Locations, California, San Bernardino Mou...","[https://www.mountainproject.com/route-guide, ...",5.7 YDS 5a French 15 Ewbanks V+ UIAA 13 ZA MVS...,5.7 YDS,"Sport, Alpine","[Sport, Alpine]",...,This high-quality route was one of the very fi...,"6 bolts, chain anchor\n(all bolts 1/2"" SS)\n* ...",This is the right-most bolted climb on the cra...,"{'Type': 'Sport, Alpine, 60 ft (18 m)', 'GPS':...","[{'photo_id': '106869997', 'parent_type': 'rou...","[{'comment_id': '105978879', 'parent_type': 'r...",14,True,11.0,12
2,central_pinnacles,105805473,https://www.mountainproject.com/route/10580547...,Coyotes at Sunset,"[All Locations, California, San Bernardino Mou...","[https://www.mountainproject.com/route-guide, ...",5.8 YDS 5b French 16 Ewbanks VI- UIAA 15 ZA HV...,5.8 YDS,"Sport, Alpine","[Sport, Alpine]",...,"This route, like it's neighbor to the right, i...","6 bolts, chain anchor\n(all bolts 1/2"" SS)\n* ...",This is the second bolted route from the left ...,"{'Type': 'Sport, Alpine, 70 ft (21 m)', 'GPS':...","[{'photo_id': '106870271', 'parent_type': 'rou...","[{'comment_id': '105978880', 'parent_type': 'r...",13,True,10.0,12
3,central_pinnacles,105874831,https://www.mountainproject.com/route/10587483...,Shoot at Will,"[All Locations, California, San Bernardino Mou...","[https://www.mountainproject.com/route-guide, ...",5.8 YDS 5b French 16 Ewbanks VI- UIAA 15 ZA HV...,5.8 YDS,"Sport, Alpine","[Sport, Alpine]",...,Starts in a recessed area and wanders up the f...,"5 bolts, chain anchor",Located on an East-facing wall around and outs...,"{'Type': 'Sport, Alpine, 45 ft (14 m)', 'GPS':...","[{'photo_id': '106816594', 'parent_type': 'rou...","[{'comment_id': '105978912', 'parent_type': 'r...",8,True,5.0,12
4,central_pinnacles,105805488,https://www.mountainproject.com/route/10580548...,Black Magic Poodle,"[All Locations, California, San Bernardino Mou...","[https://www.mountainproject.com/route-guide, ...",5.9 YDS 5c French 17 Ewbanks VI UIAA 17 ZA HVS...,5.9 YDS,"Sport, Alpine","[Sport, Alpine]",...,Start by stemming until possible to grab holds...,"8 bolts, bolted anchor/rap",This is the left-most climb on the crag and ju...,"{'Type': 'Sport, Alpine, 70 ft (21 m)', 'GPS':...","[{'photo_id': '106875312', 'parent_type': 'rou...","[{'comment_id': '105933132', 'parent_type': 'r...",9,True,6.0,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,central_pinnacles,105823204,https://www.mountainproject.com/route/10582320...,Firing Line,"[All Locations, California, San Bernardino Mou...","[https://www.mountainproject.com/route-guide, ...",5.11c YDS 6c+ French 24 Ewbanks VIII- UIAA 24 ...,5.11c YDS,"Sport, Alpine","[Sport, Alpine]",...,Easy climbing up ledges gains a stance below a...,"4 bolts, chain anchors","Located on the far right side of the wall, abo...","{'Type': 'Sport, Alpine, 50 ft (15 m)', 'GPS':...","[{'photo_id': '105932641', 'parent_type': 'rou...",[],0,False,NaN,1
96,central_pinnacles,105986456,https://www.mountainproject.com/route/10598645...,After

,output_name,area_id,url,name,breadcrumbs,breadcrumb_urls,description,getting_there,latitude,longitude,...,details_raw,child_area_urls,route_urls,photos,comments,comment_count,comments_truncated,remaining_comment_count,photo_count,route_count
0,central_pinnacles,105931166,https://www.mountainproject.com/area/105931166...,Central Pinnacles Rock Climbing,"[All Locations, California, San Bernardino Mou...","[https://www.mountainproject.com/route-guide, ...",This area truly is the heart of the Pinnacles ...,The northern parking area is located at the ed...,34.30806,-116.87845,...,"{'Elevation': '7,561 ft 2,305 m', 'GPS': '34.3...",[https://www.mountainproject.com/area/10592832...,[https://www.mountainproject.com/route/1058789...,"[{'photo_id': '106478251', 'parent_type': 'are...","[{'comment_id': '111976863', 'parent_type': 'a...",2,False,NaN,12,15
1,central_pinnacles,105928327,https://www.mountainproject.com/area/105928327...,Camp Rock Rock Climbing,"[All Locations, California, San Bernardino Mou...","[https://www.mountainproject.com/route-guide, ...",This small formation is located to the southwe...,Easily reached from the vicinty of the\nClaim ...,34.30722,-116.87897,...,"{'Elevation': '7,450 ft 2,271 m', 'GPS': '34.3...",[https://www.mountainproject.com/area/10580546...,[https://www.mountainproject.com/route/1059283...,"[{'photo_id': '109400906', 'parent_type': 'are...","[{'comment_id': '107770693', 'parent_type': 'a...",1,False,NaN,3,5
2,central_pinnacles,105807943,https://www.mountainproject.com/area/105807943...,Doc Holliday Wall Rock Climbing,"[All Locations, California, San Bernardino Mou...","[https://www.mountainproject.com/route-guide, ...","A short distance left of\nCoyote Crag\n, and j...",This formation is found 100 feet left of\nCoyo...,34.30781,-116.87833,...,"{'Elevation': '7,544 ft 2,299 m', 'GPS': '34.3...",[https://www.mountainproject.com/area/10580546...,[https://www.mountainproject.com/route/1058079...,"[{'photo_id': '106479140', 'parent_type': 'are...",[],0,False,NaN,7,7
3,central_pinnacles,105852313,https://www.mountainproject.com/area/105852313...,Gold Wall Rock Climbing,"[All Locations, California, San Bernardino Mou...","[https://www.mountainproject.com/route-guide, ...","This short, heavily featured face is just left...",Approach as for Coyote Crag and the Doc Holida...,34.30783,-116.87820,...,"{'Elevation': '7,548 ft 2,301 m', 'GPS': '34.3...",[https://www.mountainproject.com/area/10580546...,[https://www.mountainproject.com/route/1060057...,"[{'photo_id': '106029639', 'parent_type': 'are...","[{'comment_id': '105978885', 'parent_type': 'a...",1,False,NaN,5,4
4,central_pinnacles,105805460,https://www.mountainproject.com/area/105805460...,Coyote Crag Rock Climbing,"[All Locations, California, San Bernardino Mou...","[https://www.mountainproject.com/route-guide, ...","Coyote Crag, one of the most popular crags in ...",From the northern parking area hike south down...,34.30780,-116.87810,...,"{'Elevation': '7,500 ft 2,286 m', 'GPS': '34.3...",[],[https://www.mountainproject.com/route/1058054...,"[{'photo_id': '105935482', 'parent_type': 'are...","[{'comment_id': '112153906', 'parent_type': 'a...",2,False,NaN,12,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,holcomb_valley_pinnacles,120280425,https://www.mountainproject.com/area/120280425...,City Block Rock Climbing,"[All Locations, California, San Bernardino Mou...","[https://www.mountainproject.com/route-guide, ...",A small block perched on the hillside roughly ...,Hike northeast past\nEast Crags\nthen up the h...,34.30481,-116.87545,...,"{'Elevation': '7,462 ft 2,274 m', 'GPS': '34.3...",[https://www.mountainproject.com/area/10592827...,[https://www.mountainproject.com/route/1084216...,[],[],0,False,NaN,0,1
96,holcomb_valley_pinnacles,120280468,https://www.mountainproject.com/area/120280468...,Apollo Rock Rock Climbing,"[All Locations, California, San Bernardino Mou...","[https://www.mountainproject.com/ro

In [73]:
display(
    route_suggested_ratings[['route_id', 'user_id', 'user_name', 'suggested_grade']].head(10)
    if not route_suggested_ratings.empty
    else route_suggested_ratings
)
display(
    route_todos[['route_id', 'user_id', 'user_name', 'is_in_partner_finder']].head(10)
    if not route_todos.empty
    else route_todos
)
display(
    route_ticks[['route_id', 'user_id', 'user_name', 'date', 'style', 'lead_style', 'pitches', 'comment']].head(10)
    if not route_ticks.empty
    else route_ticks
)

,route_id,user_id,user_name,suggested_grade
0,105878952,15029,Joseph Stover,5.7+
1,105878952,201700713,Sweatrz Rothman,"5.7, PG13"
2,105878952,201352166,Grant Miller,5.7
3,105878952,107134199,CamBam,"5.7, PG13"
4,105878952,107163068,Jonathon French,5.7
5,105878952,106585145,Dillon J Ross,5.6
6,105878952,12549,Adam Stackhouse,5.6
7,105878952,106286160,Blair,5.6
8,105878952,106842933,Colin Schour,5.6
9,105878952,107019871,shaner,5.6


,route_id,user_id,user_name,is_in_partner_finder
0,105878952,12502.0,Graham Roff,0.0
1,105878952,105889684.0,Isaac Tait,1.0
2,105878952,106066744.0,72HW Holly,0.0
3,105878952,106196837.0,Brian Kerr,1.0
4,105878952,106462703.0,mayoyo,0.0
5,105878952,106569992.0,Shawn Adams,1.0
6,105878952,106683764.0,Markk Knowles,1.0
7,105878952,106742164.0,Claude Rogers,0.0
8,105878952,106743502.0,Cesidio Orlando,0.0
9,105878952,106846351.0,Peter Spinazze,1.0


,route_id,user_id,user_name,date,style,lead_style,pitches,comment
0,105878952,201749273.0,Patrick Lyons,2026-06-21,Lead,NaN,1,NaN
1,105878952,NaN,NaN,2026-06-19,Lead,Onsight,1,NaN
2,105878952,111261011.0,ksenn,2026-06-19,Lead,NaN,1,NaN
3,105878952,202435733.0,Anonymous,2026-06-19,TR,NaN,1,NaN
4,105878952,201324040.0,Samantha Guest,2026-06-12,TR,NaN,1,NaN
5,105878952,201103596.0,Jwalzy Campanella,2026-06-06,Lead,Pinkpoint,1,NaN
6,105878952,201769720.0,Forrest Hogue,2026-06-06,Lead,Onsight,1,NaN
7,105878952,201931511.0,A B,2026-06-06,Solo,NaN,1,NaN
8,105878952,201757474.0,Cole Baumann,2026-05-30,Lead,NaN,1,NaN
9,105878952,201666941.0,Jaden Milbrodt,2026-05-25,Lead,Redpoint,1,NaN


In [78]:
routes

,output_name,route_id,url,name,breadcrumbs,breadcrumb_urls,grade_raw,yds_grade,route_type_raw,type_tags,...,description,protection,location,details_raw,photos,comments,comment_count,comments_truncated,remaining_comment_count,photo_count
0,central_pinnacles,105878952,https://www.mountainproject.com/route/10587895...,Gold Standard,"[All Locations, California, San Bernardino Mou...","[https://www.mountainproject.com/route-guide, ...",5.6 YDS 4c French 14 Ewbanks V UIAA 12 ZA S 4b...,5.6 YDS,"Sport, Alpine","[Sport, Alpine]",...,Incut plates and edges past two bolts lead to ...,"7 bolts, chain anchors\n(all bolts 1/2"" SS)\nB...",Located on the far right side of the Gold Wall...,"{'Type': 'Sport, Alpine, 80 ft (24 m)', 'GPS':...","[{'photo_id': '106816848', 'parent_type': 'rou...","[{'comment_id': '105900371', 'parent_type': 'r...",11,True,8.0,12
1,central_pinnacles,105805463,https://www.mountainproject.com/route/10580546...,Bye Crackie,"[All Locations, California, San Bernardino Mou...","[https://www.mountainproject.com/route-guide, ...",5.7 YDS 5a French 15 Ewbanks V+ UIAA 13 ZA MVS...,5.7 YDS,"Sport, Alpine","[Sport, Alpine]",...,This high-quality route was one of the very fi...,"6 bolts, chain anchor\n(all bolts 1/2"" SS)\n* ...",This is the right-most bolted climb on the cra...,"{'Type': 'Sport, Alpine, 60 ft (18 m)', 'GPS':...","[{'photo_id': '106869997', 'parent_type': 'rou...","[{'comment_id': '105978879', 'parent_type': 'r...",14,True,11.0,12
2,central_pinnacles,105805473,https://www.mountainproject.com/route/10580547...,Coyotes at Sunset,"[All Locations, California, San Bernardino Mou...","[https://www.mountainproject.com/route-guide, ...",5.8 YDS 5b French 16 Ewbanks VI- UIAA 15 ZA HV...,5.8 YDS,"Sport, Alpine","[Sport, Alpine]",...,"This route, like it's neighbor to the right, i...","6 bolts, chain anchor\n(all bolts 1/2"" SS)\n* ...",This is the second bolted route from the left ...,"{'Type': 'Sport, Alpine, 70 ft (21 m)', 'GPS':...","[{'photo_id': '106870271', 'parent_type': 'rou...","[{'comment_id': '105978880', 'parent_type': 'r...",13,True,10.0,12
3,central_pinnacles,105874831,https://www.mountainproject.com/route/10587483...,Shoot at Will,"[All Locations, California, San Bernardino Mou...","[https://www.mountainproject.com/route-guide, ...",5.8 YDS 5b French 16 Ewbanks VI- UIAA 15 ZA HV...,5.8 YDS,"Sport, Alpine","[Sport, Alpine]",...,Starts in a recessed area and wanders up the f...,"5 bolts, chain anchor",Located on an East-facing wall around and outs...,"{'Type': 'Sport, Alpine, 45 ft (14 m)', 'GPS':...","[{'photo_id': '106816594', 'parent_type': 'rou...","[{'comment_id': '105978912', 'parent_type': 'r...",8,True,5.0,12
4,central_pinnacles,105805488,https://www.mountainproject.com/route/10580548...,Black Magic Poodle,"[All Locations, California, San Bernardino Mou...","[https://www.mountainproject.com/route-guide, ...",5.9 YDS 5c French 17 Ewbanks VI UIAA 17 ZA HVS...,5.9 YDS,"Sport, Alpine","[Sport, Alpine]",...,Start by stemming until possible to grab holds...,"8 bolts, bolted anchor/rap",This is the left-most climb on the crag and ju...,"{'Type': 'Sport, Alpine, 70 ft (21 m)', 'GPS':...","[{'photo_id': '106875312', 'parent_type': 'rou...","[{'comment_id': '105933132', 'parent_type': 'r...",9,True,6.0,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
778,pinnacles_east,106256332,https://www.mountainproject.com/route/10625633...,Central Park,"[All Locations, California, San Bernardino Mou...","[https://www.mountainproject.com/route-guide, ...",5.10a YDS 6a French 18 Ewbanks VI+ UIAA 18 ZA ...,5.10a YDS,"Sport, Alpine","[Sport, Alpine]",...,This line is in the middle of the right side o...,"5 bolts, bolted anchor",NaN,"{'Type': 'Sport, Alpine, 40 ft (12 m)', 'GPS':...","[{'photo_id': '106256340', 'parent_type': 'rou...","[{'comment_id': '106851251', 'parent_type': 'r...",3,False,NaN,2
779,pinnacles_east,106470250,https://www.mountainproject.com/route/10647025...,Wall St

## Route Map
Plot routes from the currently selected Holcomb export to a standalone HTML map using Folium with marker clustering. This keeps the original latitude/longitude intact, deduplicates routes by `route_id`, and makes overlapping route markers explorable by zooming in and spiderfying clusters.

In [128]:
import folium
from folium.plugins import MarkerCluster

map_output_name = COMMENT_AUDIT_OUTPUT_NAME or TARGET_NAME
print('Mapping output:', map_output_name)

map_routes = (
    routes[
        (routes['output_name'] == map_output_name)
        & routes['latitude'].notna()
        & routes['longitude'].notna()
    ][[
        'route_id',
        'name',
        'yds_grade',
        'route_type_raw',
        'stars_avg',
        'vote_count',
        'page_views_total',
        'latitude',
        'longitude',
    ]]
    .drop_duplicates(subset=['route_id'])
    .copy()
)

if map_routes.empty:
    print('No route coordinates available for mapping.')
else:
    map_routes['coord_key'] = list(zip(map_routes['latitude'].round(7), map_routes['longitude'].round(7)))
    map_routes['duplicate_count'] = map_routes.groupby('coord_key')['route_id'].transform('size')

    center_lat = float(map_routes['latitude'].mean())
    center_lon = float(map_routes['longitude'].mean())
    duplicate_route_count = int((map_routes['duplicate_count'] > 1).sum())
    duplicate_location_count = int((map_routes['duplicate_count'] > 1).groupby(map_routes['coord_key']).any().sum())

    route_map = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles='OpenStreetMap')
    cluster = MarkerCluster(
        name='Routes',
        options={
            'spiderfyOnMaxZoom': True,
            'showCoverageOnHover': False,
            'maxClusterRadius': 20,
        },
    ).add_to(route_map)

    for _, row in map_routes.iterrows():
        stars = 'n/a' if pd.isna(row.get('stars_avg')) else f"{float(row['stars_avg']):.2f}"
        votes = 'n/a' if pd.isna(row.get('vote_count')) else str(int(row['vote_count']))
        page_views = 'n/a' if pd.isna(row.get('page_views_total')) else str(int(row['page_views_total']))
        duplicate_note = (
            f"Shared coordinates with {int(row['duplicate_count']) - 1} other route(s)"
            if int(row['duplicate_count']) > 1
            else 'Unique route coordinates'
        )
        popup_html = '<br>'.join([
            f"<strong>{row['name']}</strong>",
            f"Grade: {row.get('yds_grade') or 'n/a'}",
            f"Type: {row.get('route_type_raw') or 'n/a'}",
            f"Stars: {stars} from {votes} votes",
            f"Page views: {page_views}",
            duplicate_note,
])

        folium.Marker(
            location=[float(row['latitude']), float(row['longitude'])],
            popup=folium.Popup(popup_html, max_width=280),
            tooltip=row['name'],
        ).add_to(cluster)

    map_output_dir = DATA_ROOT / 'maps'
    map_output_dir.mkdir(parents=True, exist_ok=True)
    map_output_path = map_output_dir / f'{map_output_name}_routes_map.html'
    route_map.save(str(map_output_path))

    print('Basemap: OpenStreetMap')
    print('Markers: exact coordinates with cluster/spiderfy behavior')
    print(f'Wrote map HTML to: {map_output_path}')
    print(f'Routes sharing exact coordinates: {duplicate_route_count} across {duplicate_location_count} coordinate groups')
    map_routes[['route_id', 'name', 'latitude', 'longitude', 'duplicate_count']].head(10)

Mapping output: holcomb_valley_pinnacles
Basemap: OpenStreetMap
Markers: exact coordinates with cluster/spiderfy behavior
Wrote map HTML to: /Users/peterwilliams/projects/mountainproject/data/maps/holcomb_valley_pinnacles_routes_map.html
Routes sharing exact coordinates: 596 across 110 coordinate groups
